# Runtime ML Integration Framework Validation
### Forex_DNN Trading Framework - Milestone 5

This notebook provides a comprehensive verification of the Runtime ML Integration Framework. It runs the entire end-to-end execution flow of the trading framework, showing how indicators, structural objects, and live market context are dynamically translated into feature vectors, evaluated through the ML Decision Engine and Signal Evaluator, and safely stored using the Trade Feature Recorder.

### Validation Objectives
1. **Equivalent Feature Extraction**: Confirm that the runtime `FeaturePipeline` generates identical features compared to the training pipeline.
2. **Registry-Driven Feature Order**: Confirm that the feature order strictly matches the `FeatureRegistry` definitions.
3. **Inference & Policy Alignment**: Verify that `MLDecisionEngine` aggregates predictions and evaluates policy rules safely.
4. **Unified Signal Evaluation**: Verify that the `SignalEvaluator` implements Shadow Mode logic where technical rules govern the trade, but ML metrics are fully logged for diagnostics.
5. **Tabular Retraining Integration**: Prove that the `TradeFeatureRecorder` records signal candidates and trade outcomes in a format directly usable for future model training.

## Section 1: System Imports

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from datetime import datetime, timedelta, timezone

# Ensure project root is in the system path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Collecting_Data.indicators import IndicatorEngine
from Market_Data_Pipeline.structure_engine import MarketStructureEngine
from Market_Data_Pipeline.supply_demand_engine import SupplyDemandEngine
from Market_Data_Pipeline.structure_graph import MarketStructureGraph
from ML.feature_registry import FeatureRegistry
from ML.feature_pipeline import FeaturePipeline
from ML.ml_decision_engine import MLDecisionEngine
from ML.trade_feature_recorder import TradeFeatureRecorder
from Strategies.signal_evaluator import SignalEvaluator, SignalEvaluation

print("All system modules loaded successfully!")

## Section 2: Synthetic Market Data Generation
We generate high-fidelity, deterministic price bars to simulate live feeds. We also add standard EMA and ATR indicators.

In [ ]:
def generate_synthetic_bars(n_bars=300, base_price=1.1200, volatility=0.0004):
    np.random.seed(42)
    end_time = datetime(2026, 7, 10, 23, 45, tzinfo=timezone.utc)
    times = [end_time - timedelta(minutes=5 * (n_bars - i - 1)) for i in range(n_bars)]
    
    prices = np.zeros(n_bars)
    prices[0] = base_price
    regime = 1
    regime_length = 0
    for i in range(1, n_bars):
        if regime_length <= 0:
            regime = np.random.choice([1, -1, 0], p=[0.35, 0.35, 0.30])
            regime_length = np.random.randint(20, 60)
        drift = regime * (volatility * 0.15) if regime != 0 else 0.0
        prices[i] = prices[i-1] + drift + np.random.normal(0, volatility)
        regime_length -= 1
        
    opens, highs, lows, closes = np.zeros(n_bars), np.zeros(n_bars), np.zeros(n_bars), np.zeros(n_bars)
    for i in range(n_bars):
        p_open = prices[i] + np.random.normal(0, volatility * 0.1)
        p_close = prices[i] + np.random.normal(0, volatility * 0.2)
        p_high = max(p_open, p_close) + abs(np.random.normal(volatility * 0.3, volatility * 0.2))
        p_low = min(p_open, p_close) - abs(np.random.normal(volatility * 0.3, volatility * 0.2))
        
        opens[i], closes[i] = p_open, p_close
        highs[i], lows[i] = max(p_high, p_open, p_close), min(p_low, p_open, p_close)
        
    return pd.DataFrame({
        'Datetime': times, 
        'Open': opens, 
        'High': highs, 
        'Low': lows, 
        'Close': closes,
        'TickVolume': np.random.randint(100, 800, n_bars).astype(float), 
        'Spread': np.ones(n_bars) * 1.5
    })

df_raw = generate_synthetic_bars()
print(f"Generated {len(df_raw)} price bars.")
display(df_raw.head())

## Section 3: Computing Indicators and Analytical Engines
We calculate technical indicators (EMA50, EMA600, EMA800, ATR14) and run the `MarketStructureEngine` and `SupplyDemandEngine` to create our `MarketStructureGraph` (MSG).

In [ ]:
# Initialize indicator engines
indicator_engine = IndicatorEngine(ema_periods=[50, 600, 800], slope_period=32)
df_indicators = indicator_engine.calculate(df_raw)

# Run analytical engines
struct_engine = MarketStructureEngine(lookback=3)
sd_engine = SupplyDemandEngine(atr_period=14, impulse_threshold=1.5)

df_struct = struct_engine.process(df_indicators)
df_final = sd_engine.process(df_struct)

# Build the MarketStructureGraph
last_row = df_final.iloc[-1]
graph = MarketStructureGraph(
    symbol="EURUSD",
    timeframe="M5",
    timestamp=pd.to_datetime(last_row["Datetime"]),
    swing_highs=[s for s in struct_engine.swings if s.level_type == 'SwingHigh'],
    swing_lows=[s for s in struct_engine.swings if s.level_type == 'SwingLow'],
    protected_high=struct_engine.protected_high,
    protected_low=struct_engine.protected_low,
    bos=list(struct_engine.bos_list),
    choch=list(struct_engine.choch_list),
    supply_zones=[z for z in sd_engine.zones if z.type == 'Supply'],
    demand_zones=[z for z in sd_engine.zones if z.type == 'Demand'],
    trend_direction="Bull" if last_row.get("trend", 0) == 1 else ("Bear" if last_row.get("trend", 0) == -1 else "Neutral"),
    atr=float(last_row.get("atr_14", 0.0001)),
    volatility=float(last_row.get("atr_14", 0.0001) * 10000.0)
)

print("MarketStructureGraph created successfully!")
print(f"Swings detected: {len(graph.swing_highs) + len(graph.swing_lows)}")
print(f"BOS detected: {len(graph.bos)}")
print(f"Supply Zones: {len(graph.supply_zones)} | Demand Zones: {len(graph.demand_zones)}")

## Section 4: Feature Pipeline Execution and Verification
We run the new `FeaturePipeline`'s `extract_runtime` method to get the clean, validated `FeatureVector` object, and we check its properties.

In [ ]:
# Initialize feature registry and pipeline
registry = FeatureRegistry(load_defaults=True)
pipeline = FeaturePipeline(registry)

# Run runtime feature extraction with mock session and strategy contexts
account_ctx = {"session": "Asian", "spread": 1.5}
strategy_ctx = {"signal_direction": 1, "signal_type": "standard"}

feature_vector = pipeline.extract_runtime(
    df=df_final,
    msg=graph,
    idx=-2, # Evaluate closed bar
    account_session_context=account_ctx,
    strategy_context=strategy_ctx
)

# Verification tests
enabled_features = registry.list_enabled()
print("=== FEATURE PIPELINE VERIFICATION ===")
print(f"1. Number of extracted features: {len(feature_vector.features)}")
print(f"2. Number of active registry features: {len(enabled_features)}")
assert len(feature_vector.features) == len(enabled_features), "Feature count mismatch!"
print("   -> SUCCESS: Feature count matches FeatureRegistry!")

# Check for order consistency
keys_in_vector = list(feature_vector.features.keys())
keys_in_registry = [f.name for f in enabled_features]
assert keys_in_vector == keys_in_registry, "Feature ordering mismatch!"
print("3. Feature order matches registry sequence: SUCCESS!")

# Check NaNs and Infinite values
has_nan = any(pd.isna(v) for v in feature_vector.features.values())
has_inf = any(np.isinf(v) for v in feature_vector.features.values() if isinstance(v, (int, float)))
print(f"4. No NaNs detected: {not has_nan}")
print(f"5. No Infinite values detected: {not has_inf}")
assert not has_nan, "NaN detected!"
assert not has_inf, "Inf detected!"

print("\nSample extracted features (first 10):")
for k, v in list(feature_vector.features.items())[:10]:
    print(f"   - {k}: {v} (Type: {type(v).__name__})")

## Section 5: ML Decision Engine Evaluation
We run the unified `MLDecisionEngine` which lazy-loads available models (or gracefully defaults with warning messages) and produces an immutable `DecisionContext` object.

In [ ]:
# Initialize MLDecisionEngine
decision_engine = MLDecisionEngine()

# Evaluate feature vector
decision_context = decision_engine.evaluate(
    symbol="EURUSD",
    timeframe="M5",
    feature_vector=feature_vector.features,
    strategy_name="MMStrategy",
    timestamp=str(graph.timestamp)
)

print("=== ML DECISION ENGINE SUMMARY ===")
print(f"Predicted State        : {decision_context.predicted_state}")
print(f"State Confidence       : {decision_context.state_confidence:.4f}")
print(f"Break Probability      : {decision_context.break_probability:.4f}")
print(f"Rejection Probability  : {decision_context.rejection_probability:.4f}")
print(f"Trade Quality Score    : {decision_context.trade_quality_score:.4f}")
print(f"Policy Recommendation  : Allow={decision_context.policy_recommendation.allow_trade}, TP={decision_context.policy_recommendation.suggested_tp_mode}")
print(f"Warnings Logged        : {decision_context.warnings}")

## Section 6: Unified Signal Evaluator
We trigger the `SignalEvaluator` on our candidate signal. Under Shadow Mode, the trade is always accepted mechanically, but full ML diagnostic information is attached to the evaluation.

In [ ]:
# Initialize SignalEvaluator in Shadow Mode
evaluator = SignalEvaluator(shadow_mode=True, ml_filtering=False)

candidate_signal = {
    "symbol": "EURUSD",
    "timeframe": "M5",
    "direction": 1,
    "signal_type": "standard",
    "technical_rules_satisfied": True
}

evaluation = evaluator.evaluate(
    strategy_name="MMStrategy",
    signal_candidate=candidate_signal,
    feature_vector=feature_vector.features,
    decision_context=decision_context,
    market_structure=graph,
    risk_state={"trading_allowed": True}
)

print("=== SIGNAL EVALUATOR SUMMARY ===")
print(f"Accepted : {evaluation.accepted} (Shadow Mode: No rejection of candidates by ML)")
print(f"Priority : {evaluation.priority}")
print(f"Reasons  : {evaluation.reasons}")

## Section 7: Tabular Trade Feature Recorder
We record the signal candidate to our tabular rolling file. Then, we simulate a trade closure and append the outcome variables using the same `signal_id`.

In [ ]:
import shutil

test_dir = "ML/recorded_features_test"
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

recorder = TradeFeatureRecorder(storage_dir=test_dir, file_format="both")

signal_id = "test-uuid-555"
timestamp_str = "2026-07-10T23:45:00Z"

# 1. Record Candidate
recorder.record_candidate(
    signal_id=signal_id,
    timestamp=timestamp_str,
    strategy="mm",
    symbol="EURUSD",
    timeframe="M5",
    direction="BUY",
    features=feature_vector.features,
    decision_context=decision_context,
    accepted=evaluation.accepted,
    reason=", ".join(evaluation.reasons)
)

# Simulate building a completed PositionLifecycle using mock outcome info
from Collecting_Data.position_lifecycle import PositionLifecycle, SignalInfo, ExecutionInfo, ManagementInfo, OutcomeInfo

sig_info = SignalInfo(signal_id=signal_id, strategy="mm", signal_category="standard", exit_profile="standard",
                      symbol="EURUSD", timeframe="M5", direction=1, signal_timestamp=timestamp_str, bar_timestamp=timestamp_str)
exec_info = ExecutionInfo(ticket=12345, magic_number=999, requested_entry=1.1210, actual_entry=1.1211, average_entry=1.1211,
                          initial_volume=0.1, remaining_volume=0.0, risk_percent=1.0, risk_amount=100.0, 
                          initial_stop_loss=1.1190, initial_take_profit=1.1250, spread=1.5, slippage=0.0001, execution_latency=0.05)
mgt_info = ManagementInfo()
out_info = OutcomeInfo(exit_timestamp="2026-07-11T00:30:00Z", average_exit_price=1.1250, close_price=1.1250, 
                       realized_profit=390.0, profit_points=39.0, profit_pips=39.0, profit_percent=3.9, 
                       r_multiple=2.0, result="WIN", strategy_reason="take_profit", broker_reason="tp", 
                       deal_count=2, partial_close_count=0, duration=2700.0, status="completed")

lifecycle = PositionLifecycle(signal=sig_info, execution=exec_info, management=mgt_info, outcome=out_info)

# 2. Append Outcome
recorder.record_outcome(signal_id=signal_id, lifecycle=lifecycle)

print("=== TRADE FEATURE RECORDER VERIFICATION ===")
# Read written files
recorded_csv_file = os.path.join(test_dir, "recorded_features_20260710.csv")
df_recorded = pd.read_csv(recorded_csv_file)
print(f"1. Found written CSV file: {os.path.exists(recorded_csv_file)}")
print(f"2. Row count: {len(df_recorded)} | Columns: {len(df_recorded.columns)}")
print(f"3. Recorded Result: {df_recorded.iloc[0]['trade_outcome']}")
print(f"4. Recorded Profit: ${df_recorded.iloc[0]['trade_profit']:.1f}")
assert len(df_recorded) == 1
assert df_recorded.iloc[0]["trade_outcome"] == "WIN"
print("   -> SUCCESS: Tabular candidate and outcome records correctly logged!")

# Clean up test recorder directory
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

## Section 8: Visualizing Price and Decoded ML Overlays
We render a clean white candlestick chart overlaying detected swings, zones, and displaying decision outputs.

In [ ]:
def plot_validation_chart(df, msg, eval_res, decision_ctx):
    df_subset = df.iloc[-80:].copy().reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_facecolor('#ffffff')
    fig.patch.set_facecolor('#ffffff')
    x = np.arange(len(df_subset))
    
    # 1. Candlestick Layer
    for i in range(len(df_subset)):
        color = '#26a69a' if df_subset.iloc[i]['Close'] >= df_subset.iloc[i]['Open'] else '#ef5350'
        ax.plot([i, i], [df_subset.iloc[i]['Low'], df_subset.iloc[i]['High']], color=color, linewidth=1.5)
        ax.add_patch(patches.Rectangle(
            (i - 0.3, min(df_subset.iloc[i]['Open'], df_subset.iloc[i]['Close'])),
            0.6,
            abs(df_subset.iloc[i]['Open'] - df_subset.iloc[i]['Close']),
            facecolor=color, edgecolor=color
        ))
        
    # 2. EMAs
    ax.plot(x, df_subset['ema_50'], color='red', alpha=0.8, linewidth=1.2, label='EMA50')
    
    # 3. Supply/Demand Zones
    for z in msg.supply_zones + msg.demand_zones:
        color = 'red' if z.type == 'Supply' else 'blue'
        rect = patches.Rectangle(
            (0, z.lower),
            len(df_subset),
            z.upper - z.lower,
            facecolor=color, edgecolor=color, alpha=0.08
        )
        ax.add_patch(rect)
        
    # 4. Annotations
    meta_text = (
        f"=== ML Diagnostic Overlays ===\n"
        f"Regime Prediction: {decision_ctx.predicted_state} ({decision_ctx.state_confidence*100:.1f}%)\n"
        f"Break Probability: {decision_ctx.break_probability*100:.1f}%\n"
        f"Trade Quality     : {decision_ctx.trade_quality_score*100:.1f}%\n"
        f"Signal Evaluation : Accepted={eval_res.accepted} (Shadow Mode Only)"
    )
    ax.text(0.02, 0.95, meta_text, transform=ax.transAxes, color='black', fontsize=11, 
            bbox=dict(facecolor='white', alpha=0.9, boxstyle='round,pad=0.5'))
    
    ax.set_title("EURUSD M5 Candlestick Chart with Runtime ML & Analytical Overlays", fontsize=14)
    ax.set_xlabel("Index", fontsize=11)
    ax.set_ylabel("Price", fontsize=11)
    ax.grid(True, color='#cccccc', alpha=0.5)
    plt.tight_layout()
    plt.show()

plot_validation_chart(df_final, graph, evaluation, decision_context)

## Section 9: Log File Integrity Check
Verify that logging entries are written correctly to the specified Logs directories.

In [ ]:
log_files = [
    "Logs/runtime_features.log",
    "Logs/decision_engine.log",
    "Logs/shadow_mode.log",
    "Logs/signal_evaluator.log"
]

print("=== RUNTIME LOG FILES CHECK ===")
for f in log_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    print(f"File: {f:<26} | Exists: {str(exists):<5} | Size: {size} bytes")